<a href="https://colab.research.google.com/github/Amirhossain959/geemap/blob/master/water_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee
!pip install -U geemap

In [ ]:
import ee
import geemap
import os
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(
    project= 'roads-and-highways',
    opt_url= 'https://earthengine-highvolume.googleapis.com'
)

In [ ]:
Map=geemap.Map()
Map

In [ ]:
roi = ee.Geometry.Polygon(
        [[[144.24193354953593, -37.819597765792125],
          [144.24193354953593, -38.54714062824357],
          [145.58775874484843, -38.54714062824357],
          [145.58775874484843, -37.819597765792125]]], None, False)
Map.addLayer(roi, {}, 'Lake Victoria')
Map.centerObject(roi, 6)

# Harmonized Sentinel-2 MSI: MultiSpectral Instrument,Level-2A (SR)\
water quality, SR data (Level-2A for Sentinel) is highly recommended because it removes the "haze" that heavily affects the Blue band (B2)\
Dataset Availability: 2017-03-28T00:00:00Z–2026-02-17T07:13:59.117000Z\
Dataset Producer: European Union/ESA/Copernicus\
Revisit Interval: 5 Days\
Bands number: 26


# Water Indices
https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/se2waq/

chlorophyll-a (Chl-a)\
cyanobacteria density- cya\
turbidity-turb\
Colored Dissolved Organic Matter=CDOM\
Total Suspended solid = tss\
DO\
pH


In [ ]:
def water_indices(img):
  cloud_mask= img.select('probability').lt(20)
  ms= img.select('B.*').multiply(0.0001)
  ndwi= ms.normalizedDifference(['B3', 'B8']).rename('ndwi')
  water= ndwi.gt(0.1)
  tss= ms.expression(
      'b7*(b5/b2)', {
          'b2': ms.select('B2'),
          'b5': ms.select('B5'),
          'b7': ms.select('B7')
      }
  ).rename('tss')
  cdom= ms.expression(
     '537 * exp(-2.93 * (B03/B04))', {
         'B03': ms.select('B3'),
         'B04': ms.select('B4')
     }
  ).rename('cdom')
  chl= ms.expression(
      '4.26 * pow((B03/B01), 3.94)', {
          'B01': ms.select('B1'),
          'B03': ms.select('B3')
      }
  ).rename('chl')
  cya= ms.expression(
      '115530.31 * pow((B03 * B04) / B02, 2.38)',{
          'B02': ms.select('B2'),
          'B03': ms.select('B3'),
          'B04': ms.select('B4')
      }
  ).rename('cya')
  turb= ms.expression(
      '8.93 * (B03/B01) - 6.39',{
          'B01': ms.select('B1'),
          'B03': ms.select('B3')
      }
  ).rename('turb')
  doc= ms.expression(
      '432 * exp(-2.24* (B03/B04))', {
          'B03': ms.select('B3'),
          'B04': ms.select('B4')
      }
  ).rename('doc')
  col= ms.expression(
      '25366 * exp(-4.53*(B03/B04))', {
          'B03': ms.select('B3'),
          'B04': ms.select('B4')
      }
  ).rename('col')
  secchi= ms.expression(
      '26.447 + (-1672.777 * B2) + (266.620 * B3) + (1402.560 * B4) + (-58.610 * B5)',{
          'B2': ms.select('B2'),
          'B3': ms.select('B3'),
          'B4': ms.select('B4'),
          'B5': ms.select('B5')
      }
  ).rename('secchi')
  ph= ms.expression(
      '12.2621 + (-246.4698 * B1) + (29.4987 * B3) + (300.0727 * B6) + (-140.2648 * B8)', {
          'B1': ms.select('B1'),
          'B3': ms.select('B3'),
          'B6': ms.select('B6'),
          'B8': ms.select('B8')
      }
  ).rename('ph')
  do= ms.expression(
      '9.2505 + (-171.0251 * B2) + (236.9708 * B4) + (76.8288 * B6) + (-150.7815 * B11)', {
          'B2': ms.select('B2'),
          'B4': ms.select('B4'),
          'B6': ms.select('B6'),
          'B11': ms.select('B11')
      }
  ).rename('do')
  return (tss
  .addBands(cdom).addBands(chl).addBands(cya)
  .addBands(turb).addBands(doc).addBands(col).addBands(secchi).addBands(ph).addBands(do)
  .updateMask(cloud_mask)
  .updateMask(water).copyProperties(img, ['system:time_start']))

In [ ]:
sen2= ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
cloud_prob= ee.ImageCollection("COPERNICUS/S2_CLOUD_PROBABILITY")
sen2= (
    sen2.linkCollection(cloud_prob, 'probability')
    # .filterDate('2022', '2025')
    .filterBounds(roi)
    .map(lambda x: water_indices(x))
    )


sen2.first().bandNames()

In [ ]:
wq_bands= sen2.first().bandNames().getInfo()
wq_bands

In [ ]:
time_start= ee.Date('2018')
time_end= ee.Date('2023')
time_diff= time_end.difference(time_start, 'month').round()
time_list= ee.List.sequence(0, time_diff).map(lambda t: time_start.advance(t, 'month'))
time_list

In [ ]:
# era5=(
#     ee.ImageCollection("ECMWF/ERA5/MONTHLY")
#     .filterDate(time_start, time_end)
#     .select(['mean_2m_air_temperature','total_precipitation'], ['temp', 'pr'])
# )
# era5.first().bandNames()

In [ ]:
climate_bands = ['aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'swe', 'tmmn', 'tmmx', 'vap', 'vpd', 'vs']
climate= (
    ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE")
    .filterDate(time_start, time_end)
    .select(climate_bands)
)

In [ ]:
all_bands= sen2.first().bandNames().getInfo()+ climate.first().bandNames().getInfo()
all_bands

In [ ]:
all_bands= ['tss','cdom','chl','cya','turb','doc','col','secchi', 'ph','do', 'aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'swe', 'tmmn', 'tmmx', 'vap', 'vpd', 'vs']

def create_homogenized_monthly_image(date):
  start_date = ee.Date(date)
  end_date = start_date.advance(1, 'month')

  # Create a template image with all expected bands filled with masked zeros, and explicitly cast to float32
  template_bands_list = []
  for b in all_bands:
    template_bands_list.append(ee.Image.constant(0).updateMask(ee.Image.constant(0)).rename(b).toFloat())
  template_image = ee.Image.cat(template_bands_list)

  # Retrieve raw monthly images, already pre-selected globally
  # And cast them to float32 upon retrieval
  wq_monthly = sen2.filterDate(start_date, end_date).median().toFloat()
  climate_monthly = climate.filterDate(start_date, end_date).median().toFloat()

  wq_bands= ['tss', 'cdom', 'chl', 'cya', 'turb', 'doc', 'col', 'secchi', 'ph', 'do']
  climate_bands= ['aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'swe', 'tmmn', 'tmmx', 'vap', 'vpd', 'vs']

    # Use ee.Algorithms.If to handle potential empty collections for dynamic data
  wq_data = ee.Image(ee.Algorithms.If(
      wq_monthly.bandNames().size().gt(0),
      wq_monthly.select(wq_bands),
      template_image.select(wq_bands)
    ))
  climate_data= ee.Image(ee.Algorithms.If(
      climate_monthly.bandNames().size().gt(0),
      climate_monthly.select(climate_bands),
      template_image.select(climate_bands)
    ))
    # Combine all data into the template image, overwriting masked zeros with actual data
  combined_image= (
      template_image
      .addBands(wq_data, None, True)
      .addBands(climate_data, None, True)
    )
  final_image= combined_image.select(all_bands).toFloat()
  return final_image.set('system:time_start', start_date.millis())

In [ ]:
collection= ee.ImageCollection(time_list.map(create_homogenized_monthly_image))
collection.first().bandNames()

In [ ]:
built= (
    ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_C")
    .select(['built_characteristics'], ['built'])
    .filterBounds(roi)
    .mode()
)
dem= (
    ee.Image("NASA/NASADEM_HGT/001")
    .select('elevation')
)

In [ ]:
collection= collection.map(
    lambda x: x.addBands(built).addBands(dem)
)

In [ ]:
ds= xr.open_dataset(
    collection,
    engine= 'ee',
    crs= 'epsg:4326',
    geometry= roi,
    scale= 0.01
)


In [ ]:
ds= ds.sortby('time') * 1
ds

In [ ]:
ds2022= ds.sel(time= slice('2022-01-01', '2022-12-31'))
ds2022.turb.plot(
    x='lon', y= 'lat', col='time', col_wrap=3,
)

In [ ]:
ds_month= ds.resample(time='M').mean('time')
ds_month_group= ds_month.groupby('time.month').mean('time').pr.plot(
    x='lon', y= 'lat', col='month', col_wrap=3,robust=True
)

In [ ]:
ds_yearly= ds.resample(time='YE').mean('time')
ds_yearly

In [ ]:
ds_yearly.turb.plot(
    x='lon', y= 'lat', col='time', col_wrap=3,
)

In [ ]:
df= ds.to_dataframe()
df

In [ ]:
df.isna().sum()

In [ ]:
def filling_missing_values(data):
  data= data.fillna(method= 'ffill')
  data= data.fillna(method= 'bfill')
  return data

df= filling_missing_values(df)
df.isna().sum()

In [ ]:
df.describe()


In [ ]:
df.info()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import time

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import time

import tensorflow as tf
from tensorflow.keras.layers import Dense, Activation
# Dense Neural Network
from tensorflow.keras.layers import Dense, Dropout
# Sequential Connection with Neural Network
from tensorflow.keras.models import Sequential
# Optimizers for Regression Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, PReLU

#from tensorflow.keras.layers import PReLU, ELU, Activation
#from tensorflow.keras.layers import Dense, LeakyReLU
from keras.layers import Dense, Activation, LeakyReLU, PReLU, ELU

In [ ]:
df.swe.value_counts()

In [ ]:
df= df.drop( columns=['swe'], axis=1)
df.columns

In [ ]:
df.corr(numeric_only=True)


In [ ]:
plt.figure(figsize=(15,15)) # Adjust figsize for a single row
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation of ph with all other variables')
plt.show()

In [ ]:
plt.figure(figsize=(15,1)) # Adjust figsize for a single row
sns.heatmap(df.corr().loc[['ph']], annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation of ph with all other variables')
plt.show()

In [ ]:
plt.figure(figsize=(15,1)) # Adjust figsize for a single row
sns.heatmap(df.corr().loc[['do']], annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation of ph with all other variables')
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.linear_model import LogisticRegression, Perceptron, SGDClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import time

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
import time

import tensorflow as tf
from tensorflow.keras.layers import Dense, Activation
# Dense Neural Network
from tensorflow.keras.layers import Dense, Dropout
# Sequential Connection with Neural Network
from tensorflow.keras.models import Sequential
# Optimizers for Regression Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, PReLU

#from tensorflow.keras.layers import PReLU, ELU, Activation
#from tensorflow.keras.layers import Dense, LeakyReLU
from keras.layers import Dense, Activation, LeakyReLU, PReLU, ELU

In [ ]:
df.columns

In [ ]:
X = df[['aet', 'def', 'pdsi', 'pet', 'pr', 'ro', 'soil', 'srad', 'tmmn', 'tmmx',
       'vap', 'vpd', 'vs']]
y = df['do']
X

In [ ]:
# standard scaler
scaler= StandardScaler()
X= scaler.fit_transform(X)
y

In [ ]:
# Assume 'target' is the column you want to predict
X = X
y = y

# Encode labels if they are not numerical
le = LabelEncoder()
y = le.fit_transform(y)

# Split the dataset into training, testing, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.3, random_state=42)

# Define models to be evaluated
models = {
    'Logistic Regression': LogisticRegression(),
    'Support Vector Machines': SVC(),
    'Linear SVC': LinearSVC(),
    'k-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Perceptron': Perceptron(),
    'Stochastic Gradient Descent': SGDClassifier(),
    'Decision Tree Classifier': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'XGBClassifier': XGBClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'AdaBoostClassifier': AdaBoostClassifier(),
}

In [ ]:
# Initialize an empty list to store the results
results_list = []

# Training and evaluating the models
for model_name, model in models.items():
    start_time = time.time()
    model.fit(X_train, y_train)

    # Training set
    y_train_pred = model.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred, average='weighted')
    train_recall = recall_score(y_train, y_train_pred, average='weighted')
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')

    # Testing set
    y_test_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred, average='weighted')
    test_recall = recall_score(y_test, y_test_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')

    # Validation set
    y_val_pred = model.predict(X_val)
    val_accuracy = accuracy_score(y_val, y_val_pred)
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_precision = precision_score(y_val, y_val_pred, average='weighted')
    val_recall = recall_score(y_val, y_val_pred, average='weighted')
    val_f1 = f1_score(y_val, y_val_pred, average='weighted')

    # Store results in the list
    results_list.append({
        'Model': model_name,
        'Training Accuracy': train_accuracy,
        'Testing Accuracy': test_accuracy,
        'Validation Accuracy': val_accuracy,
        'CPU times': time.time() - start_time,
        'MSE': val_mse,  # Use validation MSE as the overall MSE
        'Precision': train_precision,
        'Recall': train_recall,
        'F1 Score': train_f1
    })

# Convert the results list to a DataFrame
results_df = pd.DataFrame(results_list)

# Display the results
results_df

In [ ]:
# Split the dataset into training, testing, and validation sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
X_train.shape[1]

In [ ]:
def build_model():
    model = Sequential()
    model.add(Dense(units = 64, activation = 'relu', input_dim = X_train.shape[1]))
    model.add(Dense(units = 128, activation = 'relu'))
    model.add(Dense(units = 128, activation = 'relu'))
    # Output Layer - For multi-class classification
    model.add(Dense(units = num_classes, activation = 'softmax')) # Changed to num_classes and softmax

    optimizers = Adam(learning_rate = 0.001)

    # Model Compiler - Error Function for multi-class with integer labels = 'sparse_categorical_crossentropy'
    model.compile(loss = 'sparse_categorical_crossentropy', optimizer = optimizers, metrics = ['accuracy']) # Changed loss and metrics

    return model

In [ ]:
model1 = build_model()

In [ ]:
model1.summary()

In [ ]:
history1= model1.fit(X_train, y_train, epochs= 100, batch_size=32, validation_data=(X_val, y_val))


In [ ]:
pd.DataFrame(history1.history)[['accuracy', 'val_accuracy']].plot()

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_probabilities = model1.predict(X_test)
y_pred_classes = np.argmax(y_pred_probabilities, axis=1) # Get predicted class for each sample

# Evaluate using classification metrics
test_accuracy = accuracy_score(y_test, y_pred_classes)
print(f'Test Accuracy: {test_accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_classes, zero_division=0))